# HDD 18 — Frozen five-model Transformer scoring and calibration

This notebook reloads the **completed Notebook 17 checkpoint**. It builds valid 30-day windows for Notebook 15's calibration/evaluation drives, hides four final-day SMART readings (5, 194, 197, 198) and obtains their reconstruction errors in **one deterministic forward pass**. It scores three *separate* alternatives: temperature error, mean sector-counter error, and the mean of all four errors. Model-specific 99.8th-percentile thresholds are fitted on **August 16–31 calibration drives only**. Q3 evaluation-drive scores are saved without opening failure labels. No method is selected by event outcomes, and Q4/June remain closed.

All 61 dated outputs are cached with checkpoint/protocol hashes. If CPU inference is interrupted, rerun all cells; already verified daily score files are reused. This is an experimental scoring rule: anomaly score is not failure probability, no temporal regime has been validated here, and all event claims require the next matched evaluation.

In [1]:
from pathlib import Path
from collections import deque
from datetime import datetime, timezone
import hashlib,json,re,zipfile
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader,TensorDataset
try:
    from IPython.display import display
except ImportError:
    display=print
COMPONENT_ROOT=Path(r'D:/SLIIT/Y4S1/RP/Research/J26-DS-342-/01_component_1_behavioral_drift_genai')
RUN_ID='single_model_q1q2_2021_v1'
AUDIT_ID='q3_multimodel_event_coverage_v1'
PILOT_ID='q3_five_model_isolation_forest_v1'
TRAIN_ID='q3_five_model_masked_transformer_v1'
SCORE_ID='q3_five_model_transformer_scoring_v1'
AUDIT=COMPONENT_ROOT/'results'/RUN_ID/AUDIT_ID
PILOT=COMPONENT_ROOT/'results'/RUN_ID/PILOT_ID
TRAIN=COMPONENT_ROOT/'results'/RUN_ID/TRAIN_ID
TRAINED=COMPONENT_ROOT/'models/trained'/RUN_ID/TRAIN_ID
OUT=COMPONENT_ROOT/'results'/RUN_ID/SCORE_ID
DAY_OUT=OUT/'daily_scores'
for folder in [OUT,DAY_OUT]:folder.mkdir(parents=True,exist_ok=True)
Q3_ZIP=COMPONENT_ROOT/'data/raw/data_Q3_2021.zip'
def sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(4*1024*1024),b''):h.update(block)
    return h.hexdigest()
def load_json(path):
    assert path.is_file(),f'Missing input: {path}'
    return json.loads(path.read_text())
audit=load_json(AUDIT/'multimodel_coverage_complete.json')
pilot=load_json(PILOT/'pilot_complete.json')
trained=load_json(TRAIN/'encoder_training_complete.json')
assert all(x['analysis_complete'] for x in [audit,pilot,trained])
assert not audit['future_test_outcomes_opened'] and not pilot['future_test_outcomes_opened']
assert not trained['future_test_outcomes_opened'] and not trained['failure_labels_used']
reservation_path=COMPONENT_ROOT/audit['split_path']
assert sha256(reservation_path)==audit['split_sha256']
assert sha256(Q3_ZIP)==audit['settings']['archive_sha256']
reservation=load_json(reservation_path)
models=list(reservation['models']);assert len(models)==5
model_index={m:i for i,m in enumerate(models)}
train_settings=load_json(TRAIN/'settings.json')
prep_path=TRAINED/'preprocessing.json';prep=load_json(prep_path)
checkpoint_path=TRAINED/'best_model.pt'
assert sha256(prep_path)==trained['preprocessing_sha256']
assert sha256(checkpoint_path)==trained['best_checkpoint_sha256']
assert train_settings['models']==prep['models']==models
assert train_settings['features']==prep['features']
settings15=load_json(PILOT/'settings.json')
assert sha256(PILOT/'settings.json')==pilot['settings_sha256']
SALT=settings15['salt']
def group_for(serial):
    x=int(hashlib.sha256((SALT+'|'+str(serial)).encode()).hexdigest()[:16],16)/16**16
    return 'fit' if x<.60 else ('calibration' if x<.80 else 'evaluation')
active={str(s):(m,group_for(s)) for m,g in reservation['models'].items() for s in g['development'] if group_for(s) in ('calibration','evaluation')}
reserved={str(s) for g in reservation['models'].values() for s in g['future_test']}
assert not set(active)&reserved
FEATURES=train_settings['features']
assert FEATURES==['smart_5_raw','smart_9_raw','smart_194_raw','smart_197_raw','smart_198_raw']
DATES=pd.date_range('2021-07-01','2021-09-30')
DAY={str(x.date()):i for i,x in enumerate(DATES)}
SETTINGS={'checkpoint_sha256':sha256(checkpoint_path),'preprocessing_sha256':sha256(prep_path),
          'reservation_sha256':sha256(reservation_path),'q3_zip_sha256':sha256(Q3_ZIP),
          'pilot_feature_manifest_sha256':sha256(PILOT/'feature_manifest.json'),
          'scoring_dates':['2021-08-01','2021-09-30'],
          'calibration_dates':['2021-08-16','2021-08-31'],
          'evaluation_role':'evaluation','batch_size':256,
          'hidden_final_day_features':['smart_5_raw','smart_194_raw','smart_197_raw','smart_198_raw'],
          'detectors':['temp_error','sector_error','joint_error'],
          'score_meanings':{'temp_error':'squared final-day 194 residual',
                            'sector_error':'mean squared final-day 5,197,198 residual',
                            'joint_error':'mean of the four hidden final-day squared residuals'},
          'quantile':.998,'alarm_rule':'strictly greater than model-specific higher-quantile threshold',
          'labels_used':False,'transition_score_enabled':False}
settings_path=OUT/'settings.json'
if settings_path.exists():assert load_json(settings_path)==SETTINGS,'Changed scoring rule: use a new SCORE_ID.'
else:settings_path.write_text(json.dumps(SETTINGS,indent=2))
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Checkpoint verified | device:',DEVICE,'| active development drives:',len(active))
print('Q4 and failure labels unopened for this scoring notebook.')

Checkpoint verified | device: cpu | active development drives: 30867
Q4 and failure labels unopened for this scoring notebook.


## 1. Restore the exact trained model and check hidden-value leakage

The architecture matches Notebook 17, including adjacent-day differences. A difference is zeroed whenever either source value is hidden. Perturbing a hidden target must therefore leave its prediction unchanged. This assertion catches a direct shortcut from targets into derived inputs.

In [2]:
class MaskedTelemetryTransformer(nn.Module):
    def __init__(self,n_models,width=64,heads=4,layers=2,feedforward=128,dropout=.1):
        super().__init__()
        self.input=nn.Linear(15,width)
        self.model_embedding=nn.Embedding(n_models,width)
        self.position=nn.Parameter(torch.zeros(1,30,width))
        nn.init.normal_(self.position,std=.02)
        block=nn.TransformerEncoderLayer(d_model=width,nhead=heads,dim_feedforward=feedforward,
                                          dropout=dropout,batch_first=True,norm_first=True)
        self.encoder=nn.TransformerEncoder(block,num_layers=layers,enable_nested_tensor=False)
        self.output=nn.Linear(width,5)
    def forward(self,x,model_id,hide):
        observed=(~hide).to(x.dtype)
        masked=x.masked_fill(hide,0)
        delta=torch.zeros_like(x)
        delta[:,1:,:]=x[:,1:,:]-x[:,:-1,:]
        delta_visible=observed.clone()
        delta_visible[:,1:,:]=observed[:,1:,:]*observed[:,:-1,:]
        delta=delta*delta_visible
        h=self.input(torch.cat([masked,observed,delta],dim=-1))
        h=h+self.model_embedding(model_id)[:,None,:]+self.position
        return self.output(self.encoder(h))

network=MaskedTelemetryTransformer(len(models),train_settings['width'],train_settings['heads'],
                                   train_settings['layers'],train_settings['feedforward'],
                                   train_settings['dropout']).to(DEVICE)
checkpoint=torch.load(checkpoint_path,map_location=DEVICE,weights_only=True)
assert checkpoint['models']==models and checkpoint['preprocessing_sha256']==sha256(prep_path)
network.load_state_dict(checkpoint['state_dict']);network.eval()
HIDDEN=[FEATURES.index(x) for x in SETTINGS['hidden_final_day_features']]
assert HIDDEN==[0,2,3,4]
# Testing all predictions, not merely the hidden output, catches leaked adjacent differences.
x=torch.arange(2*30*5,dtype=torch.float32,device=DEVICE).reshape(2,30,5)/100
ids=torch.tensor([0,1],device=DEVICE)
hide=torch.zeros_like(x,dtype=torch.bool);hide[:,-1,HIDDEN]=True
with torch.no_grad():
    original=network(x,ids,hide)
    changed=x.clone();changed[hide]+=3.0
    perturbed=network(changed,ids,hide)
    torch.testing.assert_close(original,perturbed,rtol=1e-6,atol=1e-6)
print('Frozen checkpoint loaded. Derived-input hidden-target leakage check passed.')

Frozen checkpoint loaded. Derived-input hidden-target leakage check passed.


## 2. Stream valid Q3 windows and save daily raw scores

Each record must be on the exact date, have all five finite nonnegative SMART values, and belong to a declared calibration/evaluation drive. A gap resets its window. August 1–15 scores are saved for evaluation-drive event histories; calibration drives are scored only August 16–31. The scored drive/date sets must match Notebook 15's independent feature partitions each day. A daily cache is tied to the settings and checkpoint hashes; reruns reconstruct history but skip verified inference files.

In [3]:
def normalized(windows,model_names):
    result=np.empty_like(windows,dtype=np.float32)
    logged=np.log1p(windows.astype(np.float64))
    for model in models:
        chosen=np.asarray(model_names)==model
        if chosen.any():
            center=np.asarray(prep['center'][model],dtype=np.float64)
            scale=np.asarray(prep['scale'][model],dtype=np.float64)
            result[chosen]=((logged[chosen]-center)/scale).astype(np.float32)
    assert np.isfinite(result).all()
    return result

@torch.no_grad()
def predict_errors(windows,model_names):
    if len(windows)==0:return np.empty((0,4),dtype=np.float32)
    scaled=normalized(windows,model_names)
    ids=np.array([model_index[x] for x in model_names],dtype=np.int64)
    loader=DataLoader(TensorDataset(torch.from_numpy(scaled),torch.from_numpy(ids)),
                      batch_size=SETTINGS['batch_size'],shuffle=False,num_workers=0)
    collected=[]
    for x,m in loader:
        x=x.to(DEVICE);m=m.to(DEVICE)
        hide=torch.zeros_like(x,dtype=torch.bool);hide[:,-1,HIDDEN]=True
        residual=(network(x,m,hide)[:,-1,HIDDEN]-x[:,-1,HIDDEN]).square()
        collected.append(residual.cpu().numpy())
    result=np.concatenate(collected).astype(np.float32)
    assert result.shape==(len(windows),4) and np.isfinite(result).all()
    return result

with zipfile.ZipFile(Q3_ZIP) as z:
    members={}
    for info in z.infolist():
        if info.is_dir() or '__MACOSX' in info.filename:continue
        match=re.fullmatch(r'(2021-\d{2}-\d{2})\.csv',Path(info.filename).name)
        if match and match.group(1) in DAY:
            assert match.group(1) not in members
            members[match.group(1)]=info.filename
assert set(members)==set(DAY)

history={};last={};day_hashes={}
with zipfile.ZipFile(Q3_ZIP) as z:
    for day,(date,member) in enumerate(sorted(members.items())):
        seen=set();records=[]
        with z.open(member) as f:
            for chunk in pd.read_csv(f,usecols=['date','serial_number']+FEATURES,
                                     dtype={'serial_number':str},chunksize=50000):
                part=chunk.loc[chunk.serial_number.isin(active)]
                if part.empty:continue
                assert part.date.astype(str).eq(date).all()
                serials=part.serial_number.tolist()
                assert len(serials)==len(set(serials)) and seen.isdisjoint(serials)
                seen.update(serials)
                values=part[FEATURES].apply(pd.to_numeric,errors='coerce').to_numpy(dtype=np.float64)
                for serial,v in zip(serials,values):
                    if not (np.isfinite(v).all() and (v>=0).all()):
                        history.pop(serial,None);last[serial]=day;continue
                    if last.get(serial,-2)!=day-1:history[serial]=deque(maxlen=30)
                    h=history.setdefault(serial,deque(maxlen=30));h.append(v.astype(np.float32));last[serial]=day
                    if len(h)==30 and date>='2021-08-01':
                        model,role=active[serial]
                        if role=='evaluation' or ('2021-08-16'<=date<='2021-08-31' and role=='calibration'):
                            records.append((serial,model,role,np.stack(h)))
        if date<'2021-08-01':continue
        records.sort(key=lambda r:r[0])
        keys=pd.DataFrame([(s,m,r,date) for s,m,r,w in records],columns=['serial_number','model','role','end_date'])
        expected=pd.read_parquet(PILOT/'feature_partitions'/f'{date}.parquet',
                                 columns=['serial_number','model','role','end_date'])
        expected=expected.loc[expected.role.isin(['calibration','evaluation'])].copy()
        if date<'2021-08-16' or date>'2021-08-31':expected=expected.loc[expected.role.eq('evaluation')]
        expected=expected.sort_values('serial_number').reset_index(drop=True)
        pd.testing.assert_frame_equal(keys,expected,check_dtype=False)
        out_path=DAY_OUT/f'{date}.parquet';meta_path=DAY_OUT/f'{date}.json'
        day_identity=hashlib.sha256((sha256(settings_path)+sha256(PILOT/'feature_partitions'/f'{date}.parquet')+date).encode()).hexdigest()
        if out_path.exists() and meta_path.exists():
            info=load_json(meta_path)
            assert info['identity']==day_identity and info['sha256']==sha256(out_path) and info['rows']==len(keys)
        else:
            windows=np.stack([w for s,m,r,w in records]) if records else np.empty((0,30,5),dtype=np.float32)
            errors=predict_errors(windows,keys.model.to_numpy())
            table=keys.copy()
            for j,col in enumerate(['error_5','error_194','error_197','error_198']):table[col]=errors[:,j]
            table['temp_error']=errors[:,1]
            table['sector_error']=errors[:,[0,2,3]].mean(axis=1) if len(errors) else np.array([],dtype=np.float32)
            table['joint_error']=errors.mean(axis=1) if len(errors) else np.array([],dtype=np.float32)
            table.to_parquet(out_path,index=False)
            meta_path.write_text(json.dumps({'identity':day_identity,'sha256':sha256(out_path),'rows':len(table)},indent=2))
        day_hashes[out_path.name]=sha256(out_path)
        if date.endswith(('-01','-15')):print('Frozen scored:',date,'| windows:',len(keys))
assert len(day_hashes)==61
manifest={'settings_sha256':sha256(settings_path),'checkpoint_sha256':sha256(checkpoint_path),
          'daily_score_sha256':day_hashes,'failure_labels_used':False,
          'future_test_opened':False}
manifest_path=OUT/'raw_score_manifest.json'
if manifest_path.exists():assert load_json(manifest_path)==manifest
else:manifest_path.write_text(json.dumps(manifest,indent=2))
print('Frozen raw scoring complete:',len(day_hashes),'dated files.')

Frozen scored: 2021-08-01 | windows: 15505
Frozen scored: 2021-08-15 | windows: 15525
Frozen scored: 2021-09-01 | windows: 15525
Frozen scored: 2021-09-15 | windows: 15493
Frozen raw scoring complete: 61 dated files.


## 3. Fit model-specific thresholds on calibration drives and freeze evaluation flags

All three alternatives use the predeclared higher 99.8th percentile separately for each HDD model, on August 16–31 calibration drives. Evaluation drives supply no threshold values. Report alert burden without calling every non-failure endpoint a false alert. These outputs are the exact input for the next failure-warning and sensitivity evaluation.

In [4]:
cols=['serial_number','model','role','end_date','error_5','error_194','error_197','error_198']+SETTINGS['detectors']
cal=pd.concat([pd.read_parquet(DAY_OUT/f'{date}.parquet',columns=cols) for date in DAY if '2021-08-16'<=date<='2021-08-31'],ignore_index=True)
cal=cal.loc[cal.role.eq('calibration')].copy()
assert not cal.empty and not cal.duplicated(['serial_number','end_date']).any()
thresholds={}
for model in models:
    sub=cal.loc[cal.model.eq(model)]
    assert sub.serial_number.nunique()>=100 and len(sub)>=100
    thresholds[model]={name:float(np.quantile(sub[name].to_numpy(dtype=float),SETTINGS['quantile'],method='higher')) for name in SETTINGS['detectors']}
threshold_path=OUT/'calibrated_thresholds.json'
if threshold_path.exists():assert load_json(threshold_path)==thresholds
else:threshold_path.write_text(json.dumps(thresholds,indent=2))
display(pd.DataFrame(thresholds).T)
flag_dir=OUT/'evaluation_flags';flag_dir.mkdir(exist_ok=True)
flag_hashes={};burden=[]
for date in DAY:
    if date<'2021-08-01':continue
    a=pd.read_parquet(DAY_OUT/f'{date}.parquet',columns=cols)
    a=a.loc[a.role.eq('evaluation')].copy()
    for name in SETTINGS['detectors']:
        a[f'{name}_threshold']=a.model.map(lambda m:thresholds[m][name]).astype(float)
        a[f'{name}_flag']=a[name].to_numpy(dtype=float)>a[f'{name}_threshold'].to_numpy(dtype=float)
    assert not a.duplicated(['serial_number','end_date']).any()
    path=flag_dir/f'{date}.parquet'
    if path.exists():pd.testing.assert_frame_equal(pd.read_parquet(path),a)
    else:a.to_parquet(path,index=False)
    flag_hashes[path.name]=sha256(path)
flag_manifest={'raw_score_manifest_sha256':sha256(manifest_path),'thresholds_sha256':sha256(threshold_path),
               'evaluation_flag_sha256':flag_hashes,'failure_labels_used':False}
flag_path=OUT/'evaluation_flag_manifest.json'
if flag_path.exists():assert load_json(flag_path)==flag_manifest
else:flag_path.write_text(json.dumps(flag_manifest,indent=2))
for model in models:
    for name in SETTINGS['detectors']:
        total=flagged=0;drive_ids=set()
        for date in DAY:
            if not '2021-09-01'<=date<='2021-09-30':continue
            a=pd.read_parquet(flag_dir/f'{date}.parquet',columns=['model','serial_number',f'{name}_flag'])
            sub=a.loc[a.model.eq(model)]
            total+=len(sub);flagged+=int(sub[f'{name}_flag'].sum())
            drive_ids.update(sub.loc[sub[f'{name}_flag'],'serial_number'])
        burden.append({'model':model,'detector':name,'September_scored_endpoints':total,
                       'September_alerted_endpoints':flagged,'drives_with_alert':len(drive_ids),
                       'alerts_per_1000_scored_endpoints':1000*flagged/total if total else np.nan})
burden=pd.DataFrame(burden);burden.to_csv(OUT/'unlabelled_alert_burden.csv',index=False)
display(burden)
record={'created_utc':datetime.now(timezone.utc).isoformat(),'analysis_complete':True,
        'settings_sha256':sha256(settings_path),'raw_score_manifest_sha256':sha256(manifest_path),
        'thresholds_sha256':sha256(threshold_path),'evaluation_flag_manifest_sha256':sha256(flag_path),
        'failure_labels_used':False,'calibration_drives_disjoint_from_evaluation':True,
        'future_test_opened':False,'sealed_June_test_opened':False,'transition_score_enabled':False,
        'limitations':['one shared encoder checkpoint and one final-day mask rule',
                       'thresholds are development choices, not validated false-alarm guarantees',
                       'no five-model regime discovery yet','no recorded-failure metrics in this notebook']}
(OUT/'transformer_scoring_complete.json').write_text(json.dumps(record,indent=2))
print('NOTEBOOK 18 COMPLETE — frozen Transformer scores and calibration saved.')
print('Q4 unopened. Results:',OUT)

,temp_error,sector_error,joint_error
TOSHIBA MG07ACA14TA,1.010041,0.012183,0.273190
ST12000NM0008,1.383999,0.071248,0.439409
ST4000DM000,0.813048,2.784370,2.127448
ST8000NM0055,1.287985,1.632419,1.229587
HGST HMS5C4040BLE640,4.267711,0.252240,1.090450


,model,detector,September_scored_endpoints,September_alerted_endpoints,drives_with_alert,alerts_per_1000_scored_endpoints
0,TOSHIBA MG07ACA14TA,temp_error,154890,1161,376,7.495642
1,TOSHIBA MG07ACA14TA,sector_error,154890,292,15,1.885209
2,TOSHIBA MG07ACA14TA,joint_error,154890,1046,322,6.753180
3,ST12000NM0008,temp_error,91357,341,123,3.732609
4,ST12000NM0008,sector_error,91357,340,19,3.721663
5,ST12000NM0008,joint_error,91357,317,97,3.469904
6,ST4000DM000,temp_error,91041,212,41,2.328621
7,ST4000DM000,sector_error,91041,142,28,1.559737
8,ST4000DM000,joint_error,91041,144,31,1.581705
9,ST8000NM0055,temp_error,66722,614,198,9.202362


NOTEBOOK 18 COMPLETE — frozen Transformer scores and calibration saved.
Q4 unopened. Results: D:\SLIIT\Y4S1\RP\Research\J26-DS-342-\01_component_1_behavioral_drift_genai\results\single_model_q1q2_2021_v1\q3_five_model_transformer_scoring_v1
